# Clase 011 — pathlib

**Parte 0** · `pathlib` docs + PEP 428.

> 🎯 API moderna y multiplataforma para todo lo de filesystem. Adiós a `os.path.join`.

> ⏱️ ~60 min

## ⚙️ Setup

In [ ]:
import tempfile
from pathlib import Path
import time

# Carpeta de trabajo temporal
base = Path(tempfile.mkdtemp(prefix='lab011_'))
print(f'trabajo en: {base}')

## 1️⃣ `Path` vs string

```python
# ❌ Vieja escuela
import os
path = os.path.join(os.path.expanduser('~'), 'datos', '2026', 'enero.csv')

# ✅ pathlib
from pathlib import Path
path = Path.home() / 'datos' / '2026' / 'enero.csv'
```

El operador `/` se sobreescribe en `Path` para componer rutas. **Es multiplataforma**: en Windows se renderiza con `\`, en Unix con `/`.

In [ ]:
p = Path.home() / 'datos' / '2026' / 'enero.csv'
print(f'path: {p}')
print(f'parent: {p.parent}')
print(f'name: {p.name}')
print(f'stem: {p.stem}')
print(f'suffix: {p.suffix}')
print(f'parts: {p.parts}')

## 2️⃣ Crear, leer, escribir

One-liners cubren el 90% de los casos:

In [ ]:
# Crear estructura
(base / 'subdir').mkdir(parents=True, exist_ok=True)

# Escribir texto
(base / 'hola.txt').write_text('hola mundo\nsegunda línea\n', encoding='utf-8')

# Leer texto
contenido = (base / 'hola.txt').read_text(encoding='utf-8')
print('contenido:')
print(contenido)

# Binario
(base / 'datos.bin').write_bytes(b'\x00\x01\x02\x03')
print('bytes:', (base / 'datos.bin').read_bytes())

## 3️⃣ Listar archivos

- `path.iterdir()` — listado simple (no recursivo)
- `path.glob('*.csv')` — patrón en un nivel
- `path.rglob('*.py')` — recursivo (todo el árbol)

In [ ]:
# Genera archivos de muestra
for ext in ['csv', 'csv', 'txt', 'py', 'csv']:
    nombre = f'archivo_{ext}_{int(time.time()*1000)%10000}.{ext}'
    (base / nombre).write_text(f'demo {ext}')

# Solo CSVs, ordenados por tamaño
csvs = sorted(base.glob('*.csv'), key=lambda p: p.stat().st_size, reverse=True)
for p in csvs:
    print(f'  {p.name:35s} {p.stat().st_size} bytes')

## 4️⃣ Operaciones útiles

```python
p.exists()         # bool
p.is_file()        # bool
p.is_dir()         # bool
p.absolute()       # ruta absoluta (sin resolver symlinks)
p.resolve()        # ruta absoluta + resuelve symlinks
p.unlink()         # borra archivo
p.rmdir()          # borra dir vacío
p.rename(nuevo)    # renombra/mueve
p.stat().st_size   # info filesystem (tamaño, mtime, etc.)
p.with_suffix('.json')   # cambia extensión
```

In [ ]:
# Demostración
p = base / 'hola.txt'
print(f'absolute      : {p.absolute()}')
print(f'with_suffix   : {p.with_suffix(".md")}')
print(f'with_name     : {p.with_name("otro.txt")}')
print(f'mtime         : {p.stat().st_mtime:.0f}')
print(f'size          : {p.stat().st_size} bytes')

## 5️⃣ Rutas relativas al script — `__file__`

**Problema clásico**: tu script carga `data.csv` con `pd.read_csv('data.csv')` y funciona desde el directorio del proyecto, pero falla cuando lo ejecutan desde otro lado.

**Solución**: rutas relativas al script, no al cwd:

```python
from pathlib import Path

ROOT = Path(__file__).parent
df = pd.read_csv(ROOT / 'data' / 'penguins.csv')
```

`__file__` apunta al archivo Python actual. `.parent` da su carpeta. `.resolve()` lo convierte en absoluto.

## ✅ Checklist

- [ ] Uso `Path(...) / 'sub' / 'file'` en vez de strings
- [ ] Conozco `read_text` / `write_text` / `read_bytes`
- [ ] Uso `glob` y `rglob` según el alcance
- [ ] `mkdir(parents=True, exist_ok=True)` es mi default
- [ ] Rutas relativas a `__file__`, no al cwd

## 📝 Homework

Ver `README.md`. Script `inventario.py` que recorre un directorio y produce CSV con metadata.

## 🔗 Referencias

- [`pathlib` docs](https://docs.python.org/3/library/pathlib.html)
- [PEP 428](https://peps.python.org/pep-0428/)

➡️ **Siguiente:** [012 — Logging](../012-logging/README.md)